Week 14 · Day 2 — Fine-Tune BERT for Text Classification
Why this matters

Zero-shot inference is useful, but real power comes when you adapt a pretrained model to your dataset. Today, you’ll fine-tune BERT on a classification task (e.g., sentiment analysis).

Theory Essentials

Hugging Face Datasets: ready-made corpora (e.g., IMDB, AG News).

Trainer API: handles training loop, evaluation, logging, saving.

Fine-tuning = pretrained weights + supervised task-specific head.

Training requires: dataset → tokenization → DataLoader → training loop.

With GPU it’s faster, but small runs work on CPU (just fewer epochs/samples).

In [4]:
# Setup
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Load dataset (IMDB)
dataset = load_dataset("imdb")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

tokenized = dataset.map(tokenize, batched=True)

# Keep only needed columns
tokenized = tokenized.remove_columns(["text"])
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch")

# Small subset for speed (optional)
small_train = tokenized["train"].shuffle(seed=42).select(range(500))
small_test = tokenized["test"].shuffle(seed=42).select(range(200))

# Load model
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

# Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

# Training
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    logging_dir="./logs",
    logging_steps=50,
    save_steps=5000,         # big number → disables frequent saving
    save_total_limit=1
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_test,
    compute_metrics=compute_metrics,
)

trainer.train()
metrics = trainer.evaluate()
print(metrics)


Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\AI-Mastery\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
50,0.648100


c:\AI-Mastery\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.5123270750045776, 'eval_accuracy': 0.775, 'eval_precision': 0.7628865979381443, 'eval_recall': 0.7708333333333334, 'eval_f1': 0.7668393782383419, 'eval_runtime': 20.6948, 'eval_samples_per_second': 9.664, 'eval_steps_per_second': 1.208, 'epoch': 1.0}


1) Core (10–15 min)
Task: Run the fine-tuning script above. Check training loss and evaluation metrics.

'eval_loss': 0.45807740092277527, 'eval_accuracy': 0.8, 'eval_precision': 0.7641509433962265, 'eval_recall': 0.84375, 'eval_f1': 0.801980198019802, 'eval_runtime': 20.4213, 'eval_samples_per_second': 9.794, 'eval_steps_per_second': 1.224, 'epoch': 1.0

2) Practice (10–15 min)
Task: Change num_train_epochs from 1 → 3. Compare metrics.

{'eval_loss': 0.7159731984138489, 'eval_accuracy': 0.8, 'eval_precision': 0.7857142857142857, 'eval_recall': 0.8020833333333334, 'eval_f1': 0.7938144329896907, 'eval_runtime': 22.5015, 'eval_samples_per_second': 8.888, 'eval_steps_per_second': 1.111, 'epoch': 3.0}

3) Stretch (optional, 10–15 min)
Task: Try a different dataset, e.g., AG News (load_dataset("ag_news")).
Hint: It has 4 labels instead of 2 → change num_labels=4 in the model.

In [5]:
# Setup
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Load dataset (AG News)
dataset = load_dataset("ag_news")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

tokenized = dataset.map(tokenize, batched=True)

# Keep only needed columns
tokenized = tokenized.remove_columns(["text"])
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch")

# Small subset for speed (optional)
small_train = tokenized["train"].shuffle(seed=42).select(range(500))
small_test = tokenized["test"].shuffle(seed=42).select(range(200))

# Load model (4 classes for AG News)
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=4)

# Metrics (multi-class: average="weighted")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

# Training
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    logging_dir="./logs",
    logging_steps=50,
    save_steps=5000,         # big number → disables frequent saving
    save_total_limit=1
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_test,
    compute_metrics=compute_metrics,
)

trainer.train()
metrics = trainer.evaluate()
print(metrics)


Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\AI-Mastery\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
50,0.906400


c:\AI-Mastery\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.5796127915382385, 'eval_accuracy': 0.835, 'eval_precision': 0.8455446255871789, 'eval_recall': 0.835, 'eval_f1': 0.8350351189446453, 'eval_runtime': 21.0497, 'eval_samples_per_second': 9.501, 'eval_steps_per_second': 1.188, 'epoch': 1.0}


Mini-Challenge (≤40 min)

Task: Save your fine-tuned model and reload it for inference.

trainer.save_model("./finetuned-bert")
loaded_model = AutoModelForSequenceClassification.from_pretrained("./finetuned-bert")


Acceptance Criteria: Reloaded model can classify a new sentence (positive/negative).

In [6]:
# Setup
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Load dataset (IMDB)
dataset = load_dataset("imdb")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

tokenized = dataset.map(tokenize, batched=True)

# Keep only needed columns
tokenized = tokenized.remove_columns(["text"])
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch")

# Small subset for speed (optional)
small_train = tokenized["train"].shuffle(seed=42).select(range(500))
small_test = tokenized["test"].shuffle(seed=42).select(range(200))

# Load model
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

# Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

# Training
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    logging_dir="./logs",
    logging_steps=50,
    save_steps=5000,         # big number → disables frequent saving
    save_total_limit=1
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_test,
    compute_metrics=compute_metrics,
)

trainer.train()
metrics = trainer.evaluate()
print(metrics)

# Label mapping for IMDB
id2label = {0: "negative", 1: "positive"}
label2id = {"negative": 0, "positive": 1}

# Load model with mapping
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

# After training, make predictions on test set
preds = trainer.predict(small_test)
y_pred = np.argmax(preds.predictions, axis=1)

# Show first 10 predictions with labels
for i in range(10):
    print(f"Text: {dataset['test'][i]['text'][:100]}...")
    print(f"True label: {id2label[dataset['test'][i]['label']]}")
    print(f"Predicted: {id2label[y_pred[i]]}")
    print("---")


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\AI-Mastery\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
50,0.648100


c:\AI-Mastery\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'eval_loss': 0.5123270750045776, 'eval_accuracy': 0.775, 'eval_precision': 0.7628865979381443, 'eval_recall': 0.7708333333333334, 'eval_f1': 0.7668393782383419, 'eval_runtime': 20.7276, 'eval_samples_per_second': 9.649, 'eval_steps_per_second': 1.206, 'epoch': 1.0}


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\AI-Mastery\venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Text: I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-a...
True label: negative
Predicted: positive
---
Text: Worth the entertainment value of a rental, especially if you like action movies. This one features t...
True label: negative
Predicted: positive
---
Text: its a totally average film with a few semi-alright action sequences that make the plot seem a little...
True label: negative
Predicted: negative
---
Text: STAR RATING: ***** Saturday Night **** Friday Night *** Friday Morning ** Sunday Night * Monday Morn...
True label: negative
Predicted: positive
---
Text: First off let me say, If you haven't enjoyed a Van Damme movie since bloodsport, you probably will n...
True label: negative
Predicted: negative
---
Text: I had high hopes for this one until they changed the name to 'The Shepherd : Border Patrol, the lame...
True label: negative
Predicted: positive
---
Text: Isaac Florentine has made some of the best western Martial Arts 

Notes / Key Takeaways

Hugging Face makes fine-tuning simple via Trainer.

Start with subsets for speed, scale up later.

Always monitor precision/recall/F1, not just accuracy.

IMDB = binary classification, AG News = multiclass.

Fine-tuned models can be saved and reused anywhere.

Reflection

What changes when we go from zero-shot to fine-tuned results?

Why is using a small subset useful when prototyping?

1. What changes when we go from zero-shot to fine-tuned results?
Zero-shot uses general knowledge from pretraining, so predictions may be inconsistent or off-target. Fine-tuning adapts the model to your dataset and task, giving much higher accuracy, better alignment with domain-specific language, and more reliable results.

2. Why is using a small subset useful when prototyping?
A small subset makes experiments faster, cheaper, and easier to debug. You can check that your pipeline, tokenization, and training loop work correctly without waiting hours. Once everything runs, you can scale up to the full dataset for performance.